In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import torch.nn as nn
import sklearn.preprocessing
import torch.optim as optim
import talib as ta
import os
import time
import glob
import json

**Extract Stock Data From Yfinance**
In doing so, we remove any data that fulfills the following:
1. SMA_Diff is stuck at ~1.0 for multiple days
2. Avg volume is essentially 0
3. Vol is below 0.00001

*This ensures more high-quality data for our model to train on, completing the initial data extraction*

In [2]:
os.makedirs('data_raw', exist_ok=True)

ticker_df = pd.read_csv('nasdaq_tickers.csv')
ticker_df = ticker_df.dropna(subset=['Symbol'])
market_map = dict(zip(ticker_df['Symbol'].astype(str), ticker_df['Market Category'].astype(str)))
symbols = ticker_df['Symbol'].astype(str).tolist()

for symbol in symbols:
    if any(c in symbol for c in ['^', '.', '$']): 
        continue

    file_path = f'data_raw/{symbol}.parquet'

    if os.path.exists(file_path):
        continue

    try:
        print(f'Downloading data for {symbol}...')

        data_df = yf.download(symbol, period='2y', interval='1d', progress=False)

        if data_df.empty or len(data_df) < 100:
            continue

        if isinstance(data_df.columns, pd.MultiIndex):
            data_df.columns = data_df.columns.get_level_values(0)

        df = pd.DataFrame(index=data_df.index)
        prices = data_df['Close'].values.astype(float)
        
        # Core Metrics
        df['Close'] = prices.astype(np.float32)
        df['Volume'] = data_df['Volume'].values.astype(np.float32)
        
        # Check that stock is significant enough
        if (df['Close'] * df['Volume']).mean() < 1_000_000:
            continue
        
        sma_20 = ta.SMA(prices, timeperiod=20)
        
        # Check if stock is actually moving
        sma_ratio = prices / sma_20
        if (pd.Series(sma_ratio).dropna() == 1.0).mean() > 0.8:
            continue
        
        df['EMA_10'] = (prices / ta.EMA(prices, timeperiod=10)).astype(np.float32)
        df['RSI_14'] = (ta.RSI(prices, timeperiod=14)).astype(np.float32)
        df['ROC_10'] = (ta.ROC(prices, timeperiod=10)).astype(np.float32)
        df['VOL_20'] = (ta.STDDEV(prices, timeperiod=20) / prices).astype(np.float32)
        
        # Check that stock is not mostly stationary
        if df['VOL_20'].dropna().median() < 0.0001:
            continue
        
        
        avg_volume = ta.SMA(data_df['Volume'].values.astype(float), timeperiod=20)
        df['VOLUME_20_D'] = (data_df['Volume'].values.astype(float) / avg_volume).astype(np.float32)
        df['SKEW_20'] = (data_df['Close'].rolling(window=20).skew().values.astype(float)).astype(np.float32)
        df['KURT_20'] = (data_df['Close'].rolling(window=20).kurt().values.astype(float)).astype(np.float32)
        
        # Other identifiers
        df['Symbol'] = symbol
        df['Symbol'] = df['Symbol'].astype('category')
        df['Market_Category'] = market_map.get(symbol, "Unknown")
        df['Market_Category'] = df['Market_Category'].astype('category')
        
        # 5-day forward return
        df['Target'] = (data_df['Close'].pct_change(5).shift(-5)).astype(np.float32)
        df = df.dropna()

        df.to_parquet(file_path)
        
        time.sleep(0.001)
        
    except Exception as e:
        print(f"Exception on {symbol}: {e}")

$ABL: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ABL']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


/tmp/ipykernel_65802/1383565501.py:57: RuntimeWarning: invalid value encountered in divide
  df['VOLUME_20_D'] = (data_df['Volume'].values.astype(float) / avg_volume).astype(np.float32)


/tmp/ipykernel_65802/1383565501.py:57: RuntimeWarning: invalid value encountered in divide
  df['VOLUME_20_D'] = (data_df['Volume'].values.astype(float) / avg_volume).astype(np.float32)


$ATHA: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ATHA']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


$ATMC: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ATMC']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


...

/tmp/ipykernel_65802/1383565501.py:57: RuntimeWarning: invalid value encountered in divide
  df['VOLUME_20_D'] = (data_df['Volume'].values.astype(float) / avg_volume).astype(np.float32)


$ZAZZT: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ZAZZT']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


$ZBZZT: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ZBZZT']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


$ZYXI: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['ZYXI']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$FILE: possibly delisted; no price data found  (period=2y)


$1219202521:31: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$TIME:: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$CREATION: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

4 Failed downloads:
['FILE']: possibly delisted; no price data found  (period=2y)
['1219202521:31', 'TIME:', 'CREATION']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
